# CNN for Casing Defect Detection (USIT Data)
## Neural networks for energy applications — Day 4, Exercise 4

In this exercise we use a **Convolutional Neural Network** to classify image patches from
Ultrasonic Imager Tool (USIT) casing data. At each depth the USIT measures the casing
wall around 36 or 72 azimuthal sectors, producing a 2-D image (depth × azimuth).

**Learning objectives:**
1. Load and explore well data stored in HDF5 format, and check what the labels are.
2. Visualize USIT thickness maps and the defect flags stored with them.
3. Extract fixed-size image patches and label them as defective / normal.
4. **Split by well**, so that the test measures performance on wells the model has never seen.
5. Build and train a CNN binary classifier in PyTorch, choosing the epoch and the decision
   threshold on validation wells.
6. Evaluate once on test wells with precision, recall, F1 and a confusion matrix, per well,
   and compare with what a random patch split would have reported.

**Read this first: the labels are not ground truth.** The label arrays in the file are called
`all_defects_prediction_V1` and `all_defects_prediction_V2`: they are the outputs of two versions
of an earlier automated defect-detection method. A model trained on them learns to imitate V1,
and every score in this notebook measures **agreement with V1**, not the detection of real
casing defects (Section 1.1).

**Runtime.** About 5–8 minutes on a laptop CPU: two training runs of 12 epochs each.

---
## 0 — Imports and Setup

In [ ]:
import copy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h5py

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.metrics import (
    average_precision_score, precision_recall_curve, precision_score, recall_score,
    f1_score, confusion_matrix, ConfusionMatrixDisplay, classification_report
)

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 12,
})

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
rng = np.random.default_rng(RANDOM_STATE)

# The patches are small, so a CPU is fast enough. We do not use Apple's "mps" backend here:
# the model's AdaptiveAvgPool2d maps an 8 x 9 feature map to 4 x 4, which MPS does not support.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print("Setup complete.")

---
## 1 — Load the HDF5 Dataset

The file `data_AutoDefect_USIT_UKPublic.h5` contains **43 groups**. Each group is one
**logging run**, not one well: the group name starts with the well name (for example
`30_17b-A26_dwl_...`), and several wells were logged more than once, including repeat passes
over the same depth interval. The 43 runs come from **18 wells**. Each group stores:

| Key | Shape | Description |
|-----|-------|-------------|
| `THBK` | (n_depth, n_azimuth) | Casing thickness image. Values are centred near zero (a thickness *deviation*, not absolute thickness); the file stores no units. Missing samples are coded **-999.25** (the LAS null value), not NaN |
| `TDEP` | (n_depth,) | Depth, decreasing along the array; the step (15, 30 or 60 file units) matches the `_15B`/`_30B`/`_60B` suffix of the run name |
| `CCLU` | (n_depth,) | Casing collar locator signal |
| `collars` | (n_depth,) | Collar flags (0/1) |
| `all_defects_prediction_V1` | (n_depth, n_azimuth) | Defect flags from version 1 of an earlier automated method (0 = not flagged, 1 or 2 = flagged) |
| `all_defects_prediction_V2` | (n_depth, n_azimuth) | Defect flags from version 2 of that method |

The file has no attributes: no units, no descriptions and no definition of the label values.

**Tasks:**
- Open the HDF5 file, list the runs and derive the well of each run
- Print the number of runs and wells, and check whether the file stores any attributes
- Summarize every run and decide which run, if any, to exclude

In [ ]:
# YOUR CODE HERE
DATA_PATH = "data_AutoDefect_USIT_UKPublic.h5"
NULL_VALUE = -999.25   # code for a missing THBK sample


def well_id(run_name):
    """Well name from a group name, e.g. '30_17b-A26_dwl_DWL_OTHER_46806886_15B' -> '30_17b-A26'."""
    for separator in ("__DWL", "_dwl"):
        if separator in run_name:
            return run_name.split(separator)[0]
    return run_name.rsplit("_", 1)[0]   # the two names without the pattern: drop the sampling suffix


# TODO: Open the HDF5 file and list the group (run) names
# Hint: use h5py.File(DATA_PATH, "r") and list(f.keys())
# TODO: Build well_of = {run: well_id(run)} and print the number of runs and of wells
# TODO: Count the attributes stored in the file (f.attrs, and the attrs of each group and dataset)
# TODO: Print how many runs each well has

In [ ]:
# YOUR CODE HERE
# TODO: Create a summary DataFrame df_runs with one row per run and columns:
#       well, run, n_depth, n_azimuth, depth_step (median of np.diff(TDEP)),
#       missing_frac (fraction of THBK == NULL_VALUE), thbk_p1 and thbk_p99 (1st/99th
#       percentile of the valid THBK values), v1_flagged and v2_flagged (mean of V1 > 0, V2 > 0)
# Hint: Loop over run_names and read each group's arrays with g["THBK"][:]
# TODO: Look for a run whose THBK range is very different from all the others and list it in
#       EXCLUDED_RUNS (with a comment saying why)
EXCLUDED_RUNS = []

### 1.1 What the labels are, and what they are not

- The label arrays are named `..._prediction_V1` and `..._prediction_V2`. They are the output
  of two versions of an earlier automated defect-detection method, stored as pixel flags with
  values 0, 1 and 2. The file does not say what 1 and 2 mean; below, flagged pixels are on
  average thinner than unflagged ones, and value 1 more so than value 2.
- Nothing in the file says that the flags were checked against an independent reference
  (for example inspection of recovered casing). They are **not ground truth**.
- The two versions disagree substantially (next cell).

**Consequence for every result in this notebook:** we train a CNN to reproduce V1 from the
THBK image, and precision, recall and F1 measure **agreement with V1**. They do not measure
how well real defects are found. A high score can mean the CNN copies V1's mistakes; a low
score can mean V1 used information that THBK does not show. A claim about real defect
detection would need independently verified labels.

In [ ]:
# YOUR CODE HERE
# TODO: For the included runs (skip EXCLUDED_RUNS), compute the mean THBK of the valid pixels
#       with V1 value 0, 1 and 2
# TODO: Count the pixels flagged by V1 or V2 (V > 0), and the share of them flagged by both
# Hint: accumulate sums and counts run by run; exclude THBK == NULL_VALUE

---
## 2 — Visualize USIT Images

The THBK array for each run is essentially a 2-D image where:
- **rows** = depth levels
- **columns** = azimuthal sectors around the casing

We overlay the V1 flags to see where the earlier method marked anomalies. Missing samples
(-999.25) are shown as gaps.

**Tasks:**
- Pick a run with a noticeable flagged fraction
- Plot three panels: THBK thickness, V1 flags, and an overlay

In [ ]:
# YOUR CODE HERE
# TODO: Load THBK and the V1 flags for one run (e.g. the third-highest v1_flagged in df_runs)
# TODO: Replace NULL_VALUE samples by np.nan so that they show as gaps
# TODO: Create a 1x3 subplot showing:
#   - Panel 1: THBK as imshow
#   - Panel 2: V1 flags as imshow
#   - Panel 3: Overlay of the flags on thickness
# Hint: Use np.ma.masked_where to create a transparent overlay

---
## 3 — Extract Image Patches

We slide a window of size `PATCH_HEIGHT × n_azimuth` down each run to create
fixed-size image patches. A patch is labelled **defective (1)** if more than 5% of
its pixels are flagged by V1, otherwise **normal (0)**.

- We standardize all patches to 36 azimuthal columns (down-sample 72-sector runs by
  taking every second column).
- Patches containing missing samples (-999.25) are skipped; the excluded run is left out.
- With a stride of 16 rows, neighbouring 32-row patches **overlap by half**, and repeat runs
  image the same casing again. That is why every patch keeps the name of its **well**: the
  split in Section 5 is made by well.
- Depth sampling differs between runs (see `depth_step`), so a 32-row patch covers different
  lengths of pipe in different runs. We keep this simple setup; resampling to a common depth
  step would be a sensible improvement.

**Tasks:**
- Loop over all runs (except `EXCLUDED_RUNS`), extracting patches with a sliding window
- Standardize azimuth count to 36
- Label each patch from the V1 flag fraction; also keep the V2 label and the well name
- Skip patches containing missing samples

In [ ]:
# YOUR CODE HERE
PATCH_HEIGHT = 32   # depth rows per patch
PATCH_WIDTH = 36    # standardized azimuth sectors
STRIDE = 16         # sliding window stride (neighbouring patches overlap by half)
DEFECT_THRESHOLD = 0.05  # fraction of flagged pixels to label patch as defective

# TODO: Write to_36_sectors(image) that keeps every second column of 72-sector images
# TODO: Loop over the runs in the HDF5 file, skipping EXCLUDED_RUNS
# TODO: For each run, standardize azimuth to 36 columns and set THBK == NULL_VALUE to np.nan
# TODO: Slide a window of PATCH_HEIGHT with STRIDE, extracting patches
# TODO: Skip patches containing NaN (missing samples)
# TODO: Label a patch as defective (1) if its V1 flag fraction > DEFECT_THRESHOLD, else normal (0);
#       compute the same label from V2 (labels_v2) for later reference
# TODO: Keep the well of every patch in a list `groups` (use well_of[run])
# TODO: Convert to numpy arrays; print shape, class distribution and a per-well table

---
## 4 — Visualize Sample Patches

**Tasks:**
- Show 4 normal and 4 defective patches side by side

In [ ]:
# YOUR CODE HERE
# TODO: Separate normal and defective indices using np.where
# TODO: Create a 2x4 subplot grid
# TODO: Top row: 4 random normal patches
# TODO: Bottom row: 4 random defective patches
# Hint: rng.choice(normal_idx, 4, replace=False)

---
## 5 — Split by Well, then Rebalance and Normalize Using Training Wells Only

**Why by well.** Neighbouring patches overlap by half, repeat runs image the same casing,
and patches from one well share the tool, casing and noise. A random patch split puts
near-copies of training patches into the test set and measures memory, not transfer to a new
well. We therefore split the **wells**: `GroupShuffleSplit` with `groups = well` puts about
70 % of the wells in training, 15 % in validation and 15 % in test.

- **Validation wells** choose the training epoch and the decision threshold.
- **Test wells** are used once, in Section 9, for the locked model and threshold.
- **Rebalancing** (at most 2 normal patches per defective patch) is applied to the training
  wells only; validation and test wells keep their natural prevalence.
- **Normalization** statistics (mean, std) are computed on the training patches only and then
  applied unchanged to validation and test patches.

**Tasks:**
- Split the wells into training, validation and test sets with `GroupShuffleSplit`
- Print a split table: wells, patches, defective patches and prevalence per set
- Rebalance the training patches only; normalize with training statistics
- Add channel dimension: (N, H, W) → (N, 1, H, W)

In [ ]:
# YOUR CODE HERE
# TODO: Split the WELLS (not the patches) with GroupShuffleSplit(n_splits=1, random_state=RANDOM_STATE):
#       1) test_size=0.15 on all patches with groups=groups -> dev_idx, test_idx
#       2) test_size=0.15 / 0.85 on the dev patches with groups=groups[dev_idx] -> train_idx, val_idx
#          (the inner split returns positions within dev_idx: map them back with dev_idx[...])
# TODO: Check with assert that no well appears in two sets
# TODO: Print a split table (wells, patches, defective, prevalence per set) and the
#       validation and test well names

In [ ]:
# YOUR CODE HERE
# TODO: Rebalance the TRAINING patches only: keep every defective patch and at most 2 normal
#       patches per defective patch (validation and test keep their natural prevalence)
# Hint: rng.choice on the normal training indices, then rng.permutation of the combined indices
# TODO: Compute X_mean and X_std on the training patches only
# TODO: Normalize training, validation and test patches with the TRAINING statistics
# TODO: Add the channel dimension with [:, np.newaxis, :, :]

In [ ]:
# YOUR CODE HERE
# TODO: Create TensorDatasets and DataLoaders (batch_size=64) for training, validation and test
#       (shuffle only the training loader)
# Hint: torch.tensor(X_train), torch.tensor(y_train)

---
## 6 — Build the CNN Model

Build a 3-block CNN architecture:

| Block | Layers |
|-------|--------|
| Conv Block 1 | Conv2d(1, 32, 3, pad=1) → BatchNorm → ReLU → MaxPool(2) |
| Conv Block 2 | Conv2d(32, 64, 3, pad=1) → BatchNorm → ReLU → MaxPool(2) |
| Conv Block 3 | Conv2d(64, 128, 3, pad=1) → BatchNorm → ReLU → AdaptiveAvgPool(4,4) |
| Classifier | Flatten → Linear(128×4×4, 64) → ReLU → Dropout(0.5) → Linear(64, 2) |

**Tasks:**
- Implement the `DefectCNN` class inheriting from `nn.Module`
- Print the model summary and parameter count

In [ ]:
# YOUR CODE HERE
# TODO: Define class DefectCNN(nn.Module) with __init__ and forward methods
# Hint: Use nn.Conv2d, nn.BatchNorm2d, nn.MaxPool2d, nn.AdaptiveAvgPool2d,
#       nn.Linear, nn.Dropout, F.relu


---
## 7 — Train the Model

After every epoch we score the **validation wells** with the **average precision** (AP, the
area under the precision-recall curve). AP does not need a threshold and suits a minority
class; a model with no skill scores the prevalence. We keep a copy of the weights from the
epoch with the best validation AP. The test wells are not used here.

**Tasks:**
- Define CrossEntropyLoss and Adam optimizer (lr=1e-3)
- Implement `train_one_epoch` and `predict_proba` functions
- Train for 12 epochs, tracking training loss and accuracy, validation loss and validation AP
- Keep the weights of the best validation epoch

In [ ]:
# YOUR CODE HERE
# TODO: Define criterion (nn.CrossEntropyLoss) and optimizer (optim.Adam, lr=1e-3)
# TODO: Implement train_one_epoch(model, loader, criterion, optimizer, device)
#       returning (avg_loss, accuracy)
# TODO: Implement predict_proba(model, loader, criterion, device) returning
#       (defect probabilities, targets, avg_loss)
# Hint: F.softmax(outputs, dim=1)[:, 1] is the probability of class 1 (defective)

In [ ]:
# YOUR CODE HERE
NUM_EPOCHS = 12   # classroom budget; the best epoch is chosen on the validation wells

# TODO: Training loop over NUM_EPOCHS
# TODO: After each epoch, compute validation probabilities with predict_proba and the
#       validation average precision with average_precision_score (NOT the test wells)
# TODO: Track train_loss, train_acc, val_loss, val_ap in a history dict
# TODO: Keep best_state = copy.deepcopy(model.state_dict()) for the best validation AP,
#       and reload it after training with model.load_state_dict(best_state)
# TODO: Print progress each epoch

---
## 8 — Plot Training Curves

**Tasks:**
- Plot loss curves (training and validation) in one panel
- Plot the validation AP in a second panel, with the no-skill level and the kept epoch

In [ ]:
# YOUR CODE HERE
# TODO: Create 1x2 subplot: training and validation loss; validation AP with a horizontal
#       line at the validation prevalence (no skill) and a vertical line at best_epoch

---
## 9 — Choose the Threshold on Validation Wells, then Evaluate Once on Test Wells

Training on rebalanced data shifts the predicted probabilities, so 0.5 is not a meaningful
default threshold. We take the threshold with the highest F1 score on the **validation
wells**. (In practice the threshold should follow from the cost of a missed defect versus a
false alarm.)

Then model and threshold are locked, and the **test wells** are evaluated once, at their
natural prevalence. We report the overall scores, two references for reading them (a simple
baseline that scores each patch by its thinnest pixel, and how well the **V2** labels agree
with V1 on the same patches), and the scores per test well. Remember that all these numbers
measure agreement with V1 (Section 1.1).

Reading the per-well table:
- AP measures how well the scores **rank** the patches within a well; precision and recall also
  depend on the **threshold** chosen on the validation wells. A reasonable AP with a very low
  recall means that the scores in that well sit lower than in the validation wells, so the
  threshold does not transfer. For the same reason, the pooled test AP can be lower than the AP
  of every single test well.
- If both the CNN and the thinnest-pixel baseline score close to a well's prevalence, the V1 flags
  there do not correspond to thin pixels that THBK shows (V1 may have used other inputs, or
  flagged something else). A model that sees only THBK cannot reproduce such labels.

**Tasks:**
- Choose the threshold on the validation wells (highest F1)
- Evaluate the locked model once on the test wells: AP, precision, recall, F1, classification_report
- Compare with the thinnest-pixel baseline and with V2-versus-V1 agreement
- Report prevalence, AP, precision, recall and F1 per test well
- Plot a confusion matrix

In [ ]:
# YOUR CODE HERE
# TODO: 9.1 On the VALIDATION wells: compute precision_recall_curve(val_true, val_prob),
#       the F1 score for every threshold, and set THRESHOLD to the threshold with the highest F1
# TODO: 9.2 Once, on the TEST wells: predict probabilities, apply THRESHOLD, and print the
#       prevalence, average precision, precision, recall, F1 and classification_report
# TODO: 9.3 Baseline: score each patch by -patches[idx].min(axis=(1, 2)) and print its AP on
#       validation and test wells; print the precision and recall of labels_v2 against the
#       V1 labels on the test wells
# TODO: 9.4 For each test well separately: prevalence, AP of the CNN, AP of the baseline,
#       precision, recall and F1

In [ ]:
# YOUR CODE HERE
# TODO: Compute and display the confusion matrix of the test wells at THRESHOLD
# Hint: ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Normal", "Defective"])

---
## 10 — Visualize Predictions on Sample Patches

Inspection only: nothing in this section or the next changes the model or the threshold.
(Changing them after looking at test wells would turn the test wells into validation wells.)

**Tasks:**
- Select 8 random test patches
- Show each patch with its V1 label, the predicted label and the predicted probability

In [ ]:
# YOUR CODE HERE
# TODO: Select 8 random test patches (rng.choice)
# TODO: Use the test probabilities and predictions from Section 9 (do not change the threshold)
# TODO: Show each in a subplot with the V1 label, predicted label and probability

---
## 11 — Apply the Model to a Held-out Test Well

Slide the model down a run from a **test well** (never used for training, for the epoch or for
the threshold) and build a defect probability log. We take the test well with the most flagged
patches and its longest run, and show the 500-row window with the most V1 flags.

**Tasks:**
- Load one run of a test well
- Slide the model across it with the same PATCH_HEIGHT and STRIDE, normalizing with the training statistics
- Average overlapping predictions
- Plot the thickness map alongside the V1 flag track and the predicted probability track

In [ ]:
# YOUR CODE HERE
# TODO: Pick a TEST well (e.g. the one with the most defective test patches) and its longest run
# TODO: Load THBK and V1 for that run, standardize to 36 sectors, set NULL_VALUE to np.nan
# TODO: Slide the model across it with STRIDE, normalizing each window with X_mean and X_std
#       (training statistics) and using F.softmax for probabilities; skip windows with NaN
# TODO: Average overlapping predictions (leave rows without a window as NaN)
# TODO: Create a 1x3 plot: thickness map, V1 flag fraction track, predicted probability track
#       with the THRESHOLD from Section 9

---
## 12 — Compare with a Random Patch Split

What would a random patch split have told us? We repeat the recipe (rebalance and normalize
on the training part only, same model, same epoch budget, best epoch by AP) with a **random
80/20 split of the patches of the development wells** (training + validation wells). The test
wells stay untouched.

Compare the held-out AP with the validation-well AP from Section 7 and the test-well AP
from Section 9, each against its own prevalence (the no-skill level).

**Tasks:**
- Split the development-well patches at random (80/20, stratified)
- Rebalance and normalize using the random training part only; train a fresh DefectCNN
- Put the three AP values and their prevalences in one table and explain the difference

In [ ]:
# YOUR CODE HERE
# TODO: Split dev_idx at random with train_test_split(test_size=0.2, stratify=labels[dev_idx],
#       random_state=RANDOM_STATE)
# TODO: Rebalance the random training part, compute its mean and std, and build DataLoaders
# TODO: Train a fresh DefectCNN (torch.manual_seed(RANDOM_STATE) first) for NUM_EPOCHS epochs,
#       keeping the best held-out AP
# TODO: Print a table with the AP and prevalence of: the random held-out patches, the
#       validation wells (best_ap) and the test wells (Section 9)

---
## Summary

In this exercise we:

1. **Loaded** ultrasonic casing inspection data from an HDF5 file with 43 logging runs from
   18 wells, and found that its labels are the output of an earlier automated method (V1, V2),
   not verified ground truth.
2. **Visualized** USIT thickness maps and the V1 flags, handling the -999.25 missing values.
3. **Extracted** overlapping image patches, labelled them from V1, and kept each patch's well.
4. **Split by well** into training, validation and test wells, rebalanced and normalized using
   the training wells only.
5. **Built** a 3-block CNN with BatchNorm and Dropout for binary classification.
6. **Trained** for 12 epochs, keeping the epoch with the best validation AP, and chose the
   decision threshold on the validation wells.
7. **Evaluated once** on the test wells, overall and per well, next to a simple baseline and
   the agreement between V2 and V1.
8. **Applied** the model to a held-out test well to produce a defect probability log.
9. **Compared** with a random patch split of the development wells.

**Key takeaways:**
- USIT thickness data is naturally structured as 2-D images, making CNNs a natural fit.
- Overlapping patches and repeat runs make random patch splits optimistic: split by well (or
  by the unit you will deploy on) and keep the test wells for one final check.
- Rebalancing, normalization statistics, epochs and thresholds are fitted or chosen on
  training and validation data only; test wells keep their natural prevalence.
- Results vary strongly between wells: report them per well.
- Labels produced by an earlier method limit every claim: the scores here measure agreement
  with V1, not the detection of real defects.